# Cavity number preprocessing
Students should develop a software program to preprocess an image and get it ready to perform the
OCR of the cavity number of a plastic cap.
The cap has an external tab at a fixed position in relation to the cavity number.

## Task 0: Imports and setup

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
import sys

# Only for jupyter notebook visualization
%matplotlib inline 

filename="./img/d_22.bmp"
image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
plt.imshow(image, cmap='gray', vmin=0, vmax=255)
plt.show()

## Task 1: Generate a crop of the cavity number
### 1.1: Outline the cap by generating a circle that fits the cap mouth

#### 1) First Naive solution setting min Radius and maxRadius  

In [ ]:
rows = image.shape[0]

circles = cv2.HoughCircles(image, 
                           cv2.HOUGH_GRADIENT, 
                           1, 
                           rows/32,
                           minRadius=227,
                           maxRadius=250)

if circles is not None:
    if len(circles) == 1:
        hough_circle = circles[0][0]
        center = [hough_circle[0], hough_circle[1]]
        radius = hough_circle[2]
        center_in_pixels = np.int16(np.around(center))
        radius_in_pixels = np.int16(np.around(radius))

        # Print the cap mount
        circle = plt.Circle(center, radius, color='m', fill=False)
        center_patch = plt.Circle(center, 1, color="g")
        fig,ax = plt.subplots(1)
        ax.add_patch(center_patch)
        ax.add_patch(circle)
        ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        plt.show()
    else:
        print(f"Found {len(circles)} circles, they are too much, exit")
        sys.exit(1)
else:
    print("No circles")
    sys.exit(1)


##### Using houghcircles

In [ ]:
# image = cv2.resize(image, (int(image.shape[1]/2), int(image.shape[0]/2)))
from os import listdir

mypath='./img/'
plt.figure(figsize=(40,40))
    
for i, file in enumerate(sorted(listdir(mypath))):
    if file.startswith("."):
        continue

    filename="./img/"+file

    image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
    _, image_bin = cv2.threshold(image.astype(np.uint8), 40, 255, cv2.THRESH_BINARY)
    image_bin_rot = cv2.rotate(image_bin, cv2.ROTATE_180)
    rows = image.shape[0]
    
    image_masked = np.minimum(image_bin, image_bin_rot)

    circles = cv2.HoughCircles(image_masked, 
                            cv2.HOUGH_GRADIENT, # Detection method unique that is implemented
                            1, 
                            rows/4,
                            param1=500, # Theshold on the gradient
                            param2=25) # Smaller find more false positives

    ax = plt.subplot(int(np.ceil(len(listdir(mypath))/3)), 3, i+1)
    ax.set_title(file)
    if circles is not None:
        if len(circles[0]) > 1:
            print(f"Image {file} has 2 or more circles")
        for circle in circles[0]:
            center = [circle[0], circle[1]]
            radius = circle[2]
            center_in_pixels = np.int16(np.around(center))
            radius_in_pixels = np.int16(np.around(radius))
            circle = plt.Circle(center, radius, color='m', fill=False)
            ax.add_patch(circle)
    else:
        print(f"Image {file} has no circles")   
    
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    # hist, _ = np.histogram(image.flatten(), 256, [0,256])
    # ax.stem(hist)

plt.show()

#### 2) Using minEnclosingCircle

In [ ]:
from os import listdir

mypath='./img/'
plt.figure(figsize=(40,40))
    
for i, file in enumerate(sorted(listdir(mypath))):
    if file.startswith("."):
        continue

    filename="./img/"+file

    image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
    _, image_bin = cv2.threshold(image.astype(np.uint8), 40, 255, cv2.THRESH_BINARY)
    
    contours, _ = cv2.findContours(image_bin, 1, 2)
    cnt = contours[2]
    (x,y),radius = cv2.minEnclosingCircle(cnt)
    center = (int(x),int(y))
    radius = int(radius)

    ax = plt.subplot(int(np.ceil(len(listdir(mypath))/3)), 3, i+1)
    ax.set_title(file)
    circle = plt.Circle(center, radius, color='m', fill=False)
    ax.add_patch(circle)
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)

plt.show()

Better minimum enclosing circle with some euristics

In [ ]:
mypath='./img/'
plt.figure(figsize=(40,40))
    
for i, file in enumerate(sorted(listdir(mypath))):
    if file.startswith("."):
        continue

    filename="./img/"+file
    image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
    _, image_bin = cv2.threshold(image.astype(np.uint8), 20, 255, cv2.THRESH_BINARY)
    edge_detected_image = cv2.Canny(image_bin, 600, 700) 

    ax = plt.subplot(int(np.ceil(len(listdir(mypath))/5)), 5, i+1)
    ax.set_title(file)

    contours, _ = cv2.findContours(edge_detected_image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    fin_rad = 0
    fin_center = (1,1)
    
    for cnt in contours:
        (x,y),radius = cv2.minEnclosingCircle(cnt)
        center = (int(x),int(y))
        radius = int(radius)

        if radius > fin_rad and center[0] > image.shape[0]/3 and center[0] < 3*image.shape[0]/4:
            fin_rad = radius
            fin_center = center
            
    circle = plt.Circle(fin_center, fin_rad, color='b', fill=False)
    ax.add_patch(circle)
    ax.imshow(edge_detected_image, cmap='gray', vmin=0, vmax=255)

plt.show()

#### 3) Deleting white objects around

In [ ]:
from os import listdir

mypath='./img/'
plt.figure(figsize=(40,40))
    
for i, file in enumerate(sorted(listdir(mypath))):
    if file.startswith("."):
        continue

    filename="./img/"+file

    image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
    _, image_bin = cv2.threshold(image.astype(np.uint8), 20, 255, cv2.THRESH_BINARY)
    #image_bin = image
    rows = image.shape[0]
    
    cv2.rectangle(image_bin, (0,0), (150, 600), 0, -1)
    cv2.rectangle(image_bin, (650,0), (800, 600), 0, -1)
    cv2.rectangle(image_bin, (0,0), (800, 50), 0, -1)

    circles = cv2.HoughCircles(image_masked, 
                            cv2.HOUGH_GRADIENT, # Detection method unique that is implemented
                            1, 
                            rows/8,
                            param1=500, # Theshold on the gradient
                            param2=25) # Smaller find more false positives

    ax = plt.subplot(int(np.ceil(len(listdir(mypath))/3)), 3, i+1)
    ax.set_title(file)
    if circles is not None:
        if len(circles[0]) > 1:
            print(f"Image {file} has 2 or more circles")
        for circle in circles[0]:
            center = [circle[0], circle[1]]
            radius = circle[2]
            center_in_pixels = np.int16(np.around(center))
            radius_in_pixels = np.int16(np.around(radius))
            circle = plt.Circle(center, radius, color='m', fill=False)
            ax.add_patch(circle)
    else:
        print(f"Image {file} has no circles")   

    contours, _ = cv2.findContours(image_bin, 1, 2)
    cnt = contours[-1]
    (x,y),radius = cv2.minEnclosingCircle(cnt)
    center = (int(x),int(y))
    radius = int(radius)
    circle = plt.Circle(center, radius, color='b', fill=False)
    ax.add_patch(circle)
    
    ax.imshow(image_bin, cmap='gray', vmin=0, vmax=255)
    # hist, _ = np.histogram(image.flatten(), 256, [0,256])
    # ax.stem(hist)

plt.show()

The idea work now we have to find out how to delete them

#### 4) WORKING SOLUTION Delete objects around 

In [ ]:
from os import listdir

mypath='./img/'
plt.figure(figsize=(40,40))
    
for i, file in enumerate(sorted(listdir(mypath))):
    if file.startswith("."):
        continue

    filename="./img/"+file

    image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
    _, image_bin = cv2.threshold(image.astype(np.uint8), 20, 255, cv2.THRESH_BINARY)
    edge_detected_image = cv2.Canny(image_bin, 75, 100)

    middle_line = edge_detected_image[:, int(edge_detected_image.shape[1]/2)]
    non_zero_indices = np.nonzero(middle_line)[0]
    diameter = non_zero_indices[-1]-non_zero_indices[0]
    radius = int(diameter/2)

    edge_detected_image[:, 0:int(edge_detected_image.shape[1]/2) - int(radius*1.05)] = 0
    edge_detected_image[:, int(edge_detected_image.shape[1]/2) + int(radius*1.05):] = 0

    edge_detected_image[0:int(edge_detected_image.shape[0]/2) - int(radius*1.1), :] = 0
    edge_detected_image[int(edge_detected_image.shape[0]/2) + int(radius*1.1):, :] = 0

    found = False
    par2 = 25
    while not found and par2 > 5:
        circles = cv2.HoughCircles(edge_detected_image, 
                                cv2.HOUGH_GRADIENT, # Detection method unique that is implemented
                                1, 
                                rows/4,
                                param1=500, # Theshold on the gradient
                                param2=par2)  # Smaller find more false positives
        if circles is not None and len(circles) > 0:
            found = not found
        par2 -= 5

    ax = plt.subplot(int(np.ceil(len(listdir(mypath))/5)), 5, i+1)
    ax.set_title(file)
    if circles is not None:
        if len(circles[0]) > 1:
            print(f"Image {file} has 2 or more circles")
        for circle in circles[0]:
            center = [circle[0], circle[1]]
            radius = circle[2]
            center_in_pixels = np.int16(np.around(center))
            radius_in_pixels = np.int16(np.around(radius))
            circle = plt.Circle(center, radius, color='m', fill=False)
            ax.add_patch(circle)
    else:
        print(f"Image {file} has no circles") 
    
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)

plt.show()

### 1.2 Generate a crop containing the cavity number. The crop should contain the cavity number and it should appear upright

In [ ]:
radius_inside = int(radius)
radius_outside = radius_inside + 15

fig,ax = plt.subplots(1)
circle_inside = plt.Circle(center, radius_inside, color='m', fill=False)
circle_ouside = plt.Circle(center, radius_outside, color='b', fill=False)
ax.add_patch(circle_inside)
ax.add_patch(circle_ouside)
ax.imshow(image, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
# Create a mask with the same dimensions as the image
mask = np.zeros(image.shape[:2], dtype=np.uint8)


# Draw a transparent circle on the mask
cv2.circle(mask, center_in_pixels, radius_outside, (255), thickness=-1)
# Draw a black circle on the mask
cv2.circle(mask, center_in_pixels, radius_inside, (0), thickness=-1)

# Extract the circular ROI using the mask
circular_roi = cv2.bitwise_and(image, image, mask=mask)

# Filter to delete some white areas that are not the tab
circular_roi_gauss = cv2.medianBlur(circular_roi, 11)

# Display the original image and the circular ROI
fig,ax = plt.subplots(1)
circular_roi_rgb = cv2.cvtColor(circular_roi_gauss, cv2.COLOR_BGR2RGB)
ax.imshow(circular_roi_rgb)
plt.show()

In [ ]:
circular_roi_gauss_th = cv2.threshold(circular_roi_gauss, 127, 255, cv2.THRESH_BINARY)[1]
num_labels, labels_im, stats, centroid = cv2.connectedComponentsWithStats(circular_roi_gauss_th)

if len(centroid) > 2:
    print(f"Found {len(centroid)} object in the anular region, they are too much")
    sys.exit(1)
elif len(centroid) < 2:
    print("Tab not found!")
    sys.exit(1)

# 0 is the backgorud, 1 is the tab
tab_center = centroid[1]

def imshow_components(labels):
    # Map component labels to hue val
    label_hue = np.uint8(179*labels/np.max(labels))
    blank_ch = 255*np.ones_like(label_hue)
    labeled_img = cv2.merge([label_hue, blank_ch, blank_ch])

    # cvt to BGR for display
    labeled_img = cv2.cvtColor(labeled_img, cv2.COLOR_HSV2BGR)

    # set bg label to black
    labeled_img[label_hue==0] = 0

    # center 
    center_patch = plt.Circle(tab_center, 1, color="g")


    fig,ax = plt.subplots(1)
    ax.imshow(labeled_img)
    ax.add_patch(center_patch)
    plt.show()

imshow_components(labels_im)

In [ ]:
tab_center = np.int16(np.around(tab_center))
center = np.int16(np.around(center))

# Slope of the straight line that connects the center of the tab with the center of the cap
m = (center[1] - tab_center[1]) / (center[0] - tab_center[0]) * 1.0
# Intercept
q = center[1] - m * center[0]

line = np.polynomial.polynomial.polyline(q, m) # idk why but m is always the opposite of what expected

x_val = ([*range(tab_center[0],center[0])] , [*range(center[0], tab_center[0])])[int(tab_center[0] > center[0])]
y_val = np.polynomial.polynomial.polyval(x_val, line)

fig,ax = plt.subplots(1)
ax.imshow(image, cmap='gray', vmin=0, vmax=255)
ax.plot(x_val, y_val, color="b")
ax.plot(np.ones(image.shape[0]-1)*center[0],[*range(1, image.shape[0])], color="g")
plt.show()

In [ ]:
# Find the angle between the green and the blue lines
x = np.abs(tab_center[0]-center[0])
y = np.abs(tab_center[1]-center[1])
teta = np.arctan(x/y)

# Form rad to deg
rotation = (teta * 180 / np.pi)

# Looking at the slope we choose the direction of the rotation
if m > 0 :
    rotation = rotation * -1

# Looking at the tab position we choose if we have to overturn
if tab_center[1] > image.shape[0] / 2:
    rotation += 180


image_center = tuple(np.array(image.shape[1::-1]) / 2)
rot_mat = cv2.getRotationMatrix2D(image_center, rotation, 1.0)
image_with_vertical_tab = cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)
fig,ax = plt.subplots(1)
ax.imshow(image_with_vertical_tab, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
cavity_numer_crop = image_with_vertical_tab.copy()[100:170, 310:450]
plt.imshow(cavity_numer_crop, cmap='gray', vmin=0, vmax=255)
plt.show()

#### More robust cropping

In [ ]:
print(center)
cavity_numer_crop = image_with_vertical_tab.copy()[center[1] - int(radius*0.8) : center[1] - int(radius*0.5), center[0] - int(radius*0.4) : center[0] + int(radius*0.4)]
plt.imshow(cavity_numer_crop, cmap='gray', vmin=0, vmax=255)
plt.show()


## 2 Apply a polar transform

In [ ]:
flags = cv2.INTER_CUBIC | cv2.WARP_FILL_OUTLIERS | cv2.WARP_POLAR_LINEAR
polar_image = cv2.warpPolar(image_with_vertical_tab, image_with_vertical_tab.shape, image_center, 360, flags)
fig,ax = plt.subplots(1)
ax.imshow(polar_image, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
rotated_polar_image = cv2.rotate(polar_image, cv2.ROTATE_90_COUNTERCLOCKWISE)
fig,ax = plt.subplots(1)
ax.imshow(rotated_polar_image, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
rect_cavity_numer_crop = rotated_polar_image.copy()[300:360, 520:640]
cv2.imwrite("result.png", rect_cavity_numer_crop)
plt.imshow(rect_cavity_numer_crop, cmap='gray', vmin=0, vmax=255)
plt.show()

## 4: Not required but apply ocr

In [ ]:
from PIL import Image
import pytesseract
pytesseract.image_to_string(
    Image.open('/Users/micheletagliani/Developer/Python/CoputerVision/CvCavityNumberPreprocessing/result.png'),
    config="--psm 7")